# Concurrency, Parallelism & Asyncio (5+ Years Interview Guide)
Exhaustive revision guide to threading.Thread, threading.Lock, ThreadPoolExecutor, ProcessPoolExecutor, async/await coroutines, and asyncio.gather() on transaction batches.

### Key 5-Year Interview Concepts Covered:
- **Multi-Threading & Thread Synchronization**: Dedicated cell for `threading.Thread`, `threading.Lock`, and GIL constraints.
- **Thread & Process Pools**: Dedicated cell for `concurrent.futures.ThreadPoolExecutor` and `ProcessPoolExecutor`.
- **Asynchronous Event Loop Programming**: Dedicated cell for `async def`, `await`, `asyncio.run()`, and `asyncio.gather()`.

This interactive revision guide uses `data/raw_transactions.csv` with individual dedicated cells per method.

In [1]:
# Setup imports & dataset loading from raw_transactions.csv
import csv
import sys
import time
import os
import functools
import contextlib
import asyncio
import threading
from dataclasses import dataclass
from typing import List, Dict, Optional, Union, Protocol

csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
transactions = []
with open(csv_path, mode='r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        transactions.append(row)

print(f"Python Version: {sys.version.split()[0]}")
print(f"Loaded {len(transactions)} transaction records from {csv_path}")

Python Version: 3.12.7
Loaded 15000 transaction records from data/raw_transactions.csv


### Multi-Threading with `threading.Thread` & `threading.Lock`
**Explanation**: CPython GIL allows only one thread to execute Python bytecode at a time. Threads are ideal for I/O-bound tasks. Locks prevent race conditions on shared memory.

**Syntax**: `threading.Thread(target=fn)` / `with lock: balance += amt`

In [2]:
shared_balance = 0.0
balance_lock = threading.Lock()

def deposit(amt):
    global shared_balance
    with balance_lock:
        shared_balance += amt

threads = [threading.Thread(target=deposit, args=(float(transactions[i]['transaction_amount']),)) for i in range(5)]
for t in threads: t.start()
for t in threads: t.join()
print(f'Thread-Safe Consolidated Balance: ${shared_balance:,.2f}')

Thread-Safe Consolidated Balance: $3,086.87


### High-Level Execution Pools: `ThreadPoolExecutor`
**Explanation**: `concurrent.futures.ThreadPoolExecutor` provides pool management, mapping tasks across worker threads without manual thread lifecycle management.

**Syntax**: `with ThreadPoolExecutor(max_workers=4) as executor: results = executor.map(fn, items)`

In [3]:
from concurrent.futures import ThreadPoolExecutor
def enrich_tx(tx):
    return f"{tx['transaction_id']} (Audited: {tx['card_type']})"

with ThreadPoolExecutor(max_workers=4) as pool:
    audited_results = list(pool.map(enrich_tx, transactions[:4]))
print('Parallel ThreadPool Enriched Results:', audited_results)

Parallel ThreadPool Enriched Results: ['TX110686 (Audited: Visa)', 'TX107170 (Audited: MasterCard)', 'TX108328 (Audited: Discover)', 'TX108563 (Audited: Amex)']


### Asynchronous Event Loop: `async def`, `await`, and `asyncio.gather()`
**Explanation**: Cooperative non-blocking concurrency on a single thread. Coroutines yield control to the event loop during I/O operations via `await`.

**Syntax**: `async def main(): await asyncio.gather(t1, t2)`

In [4]:
async def simulate_fraud_api_call(tx_id):
    await asyncio.sleep(0.01) # Non-blocking async I/O sleep
    return f'{tx_id}: CLEARED'

async def run_async_batch():
    tasks = [simulate_fraud_api_call(t['transaction_id']) for t in transactions[:4]]
    return await asyncio.gather(*tasks)

async_results = asyncio.run(run_async_batch())
print('Asynchronously Processed Transactions:', async_results)

Asynchronously Processed Transactions: ['TX110686: CLEARED', 'TX107170: CLEARED', 'TX108328: CLEARED', 'TX108563: CLEARED']


## Section: Senior Fintech Interview Scenarios (5+ Years Experience)

### Q1: When to choose Threading vs Multiprocessing vs Asyncio
**Explanation**: Explain trade-offs: (1) Threading for I/O-bound legacy blocking libraries; (2) Multiprocessing to bypass GIL for heavy CPU compute; (3) Asyncio for high-concurrency network I/O (10,000+ simultaneous web sockets).

**Syntax**: `ProcessPoolExecutor` for CPU vs `asyncio` for Network I/O

In [5]:
print('CPU-Bound: Multiprocessing (separate OS processes).')
print('I/O-Bound High Concurrency: Asyncio (event loop coroutines).')
print('I/O-Bound Blocking APIs: Threading.')

CPU-Bound: Multiprocessing (separate OS processes).
I/O-Bound High Concurrency: Asyncio (event loop coroutines).
I/O-Bound Blocking APIs: Threading.
